# Interval Memory for Long Video — GPU Ingest

Runs the **ingest half** of the system: video frames → SigLIP tagging →
selective Qwen2-VL escalation → intervals → resolved memory → a committed
**feature pack**.

Everything downstream of this notebook is CPU-only. You run this once per video,
keep the pack, commit it, and reviewers reproduce the demo with no GPU.

## Running it

The notebook auto-detects its environment (`ON_KAGGLE` in section 1) and derives
every path from a single `WORKDIR`, so the same file runs in both places
unchanged.

**On Kaggle / Colab** — turn on the GPU accelerator (T4 is plenty) and enable
Internet, which is needed to pull model weights from Hugging Face. Section 1
clones the repo from GitHub, because there is no shell before the notebook starts.

**On an SSH box** — you have a shell, so use it. Install torch deliberately (the
default PyPI wheel is the ~2.5 GB CUDA build), then start Jupyter from *inside*
the checkout so section 3 finds the source with no configuration:

```bash
git clone https://github.com/Chgauravpc/Vidmm.git ~/video-memory
cd ~/video-memory

python3.11 -m venv ~/vmem && source ~/vmem/bin/activate
pip install torch --index-url https://download.pytorch.org/whl/cu121   # or /cpu
pip install "transformers>=4.45" accelerate opencv-python-headless numpy jupyter

jupyter lab --no-browser --port 8888        # then tunnel: ssh -L 8888:localhost:8888
```

Start Jupyter from within the venv — section 4 shells out with `sys.executable`
to run the test batteries, so the kernel's interpreter is the one that gets
validated. A kernel from a different environment means section 4 checks a Python
that section 6 never uses.

The memory core needs Python **3.9+** (`from __future__ import annotations`
throughout); 3.11 is recommended only because it sits comfortably inside the
torch / transformers wheel matrix.

Outputs land in `WORKDIR` (`~/vmem-out` on a server, `/kaggle/working` on Kaggle).
Nothing is written into the repo.

## 1. Config

In [ ]:
import os

# --- environment ---------------------------------------------------------
# This notebook runs unchanged on Kaggle/Colab and on a plain SSH box. The only
# real difference is where scratch space lives, so every output path below is
# derived from one root rather than hardcoded to /kaggle/working.
ON_KAGGLE = os.path.isdir("/kaggle/working")
WORKDIR   = "/kaggle/working" if ON_KAGGLE else os.path.expanduser("~/vmem-out")
os.makedirs(WORKDIR, exist_ok=True)

# --- where the source code comes from ------------------------------------
# Kaggle has no shell before the notebook starts, so it clones from GitHub.
# On a server you already have a checkout: leave both of these empty and the
# next section walks up from the working directory to find it. That also lets
# you run an unmerged branch without pushing anything first.
REPO_URL   = "https://github.com/Chgauravpc/Vidmm" if ON_KAGGLE else ""
SOURCE_DIR = "/kaggle/input/video-memory" if ON_KAGGLE else ""   # "" = auto-detect

# --- the video to ingest -------------------------------------------------
# Absolute path to an .mp4. Empty falls back to the synthetic clip, which proves
# the pipeline runs end to end but produces meaningless numbers - see section 5
# for how to get footage that actually matches the prompt bank.
VIDEO_PATH = ""

# --- ingest settings -----------------------------------------------------
SAMPLE_FPS   = 1.0   # frames sampled per second of video
TAU          = 2.0   # occlusion tolerance (s) for the gap rule
MAX_FRAMES   = 300   # cap for a first smoke run; set None for the full video
MIN_DURATION = 0.0   # drop resolved fragments shorter than this
STAGE3       = True  # False = ablation A-noVLM (SigLIP only)

OUT_PACK = os.path.join(WORKDIR, "pack")

print("ON_KAGGLE:", ON_KAGGLE)
print("WORKDIR  :", WORKDIR)

## 2. Install dependencies

In [ ]:
# Deps are installed with pip via subprocess, not the `!pip` magic, so this cell
# also works under `jupyter nbconvert --execute` and plain `python`.
import importlib.util, subprocess, sys


def _have(mod):
    return importlib.util.find_spec(mod) is not None


# torch is NOT auto-installed. Kaggle ships it; a bare server does not, and the
# default PyPI wheel is the ~2.5 GB CUDA build - which is the wrong download on
# a CPU box and the wrong CUDA version on plenty of GPU boxes. Fail with the
# exact command instead of guessing.
if not _have("torch"):
    raise RuntimeError(
        "torch is not installed. Install it deliberately so you get the right build:\n"
        "  GPU box : pip install torch --index-url https://download.pytorch.org/whl/cu121\n"
        "  CPU only: pip install torch --index-url https://download.pytorch.org/whl/cpu"
    )


def _transformers_too_old():
    if not _have("transformers"):
        return True
    import transformers as _tf
    try:
        return tuple(int(p) for p in _tf.__version__.split(".")[:2]) < (4, 45)
    except ValueError:
        return False  # a dev/rc string we cannot parse; assume the user knows


# Qwen2-VL needs transformers >= 4.45; SigLIP is well supported there too.
need = []
if _transformers_too_old():   need.append("transformers>=4.45")
if not _have("accelerate"):   need.append("accelerate")
if not _have("cv2"):          need.append("opencv-python-headless")

if need:
    print("installing:", need)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *need], check=True)
else:
    print("dependencies already satisfied")

import transformers, torch
print("python      ", sys.version.split()[0])
print("transformers", transformers.__version__)
print("torch       ", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("no CUDA - ingest runs on CPU. Slow, but x_realtime then measures the "
          "on-device number the plan actually wants (section 5.3).")

## 3. Get the source code

In [ ]:
import os, sys, shutil, subprocess


def _find_checkout(start):
    """Walk up from `start` looking for a directory that holds src/video_memory."""
    d = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(d, "src", "video_memory")):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return None
        d = parent


if REPO_URL:
    WORK = os.path.join(WORKDIR, "video-memory")
    if os.path.exists(WORK):
        shutil.rmtree(WORK)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], check=True)
    SRC = os.path.join(WORK, "src")
elif SOURCE_DIR:
    # An explicit local checkout, or a read-only Kaggle dataset. Either way the
    # package only needs to be importable, not writable.
    candidate = os.path.join(SOURCE_DIR, "src")
    SRC = candidate if os.path.isdir(candidate) else SOURCE_DIR
else:
    # Server default: you are running the notebook from inside the checkout.
    repo = _find_checkout(os.getcwd())
    if repo is None:
        raise RuntimeError(
            f"""no checkout found at or above {os.getcwd()!r}
Start Jupyter from somewhere inside the repo, or set one of these in section 1:
  SOURCE_DIR = "/path/to/video-memory"     # use a local checkout
  REPO_URL   = "https://github.com/..."    # clone it instead"""
        )
    SRC = os.path.join(repo, "src")
    print("found checkout:", repo)

# Fail with a useful message rather than a bare ModuleNotFoundError three lines
# later. The common failure is a source path that points somewhere plausible but
# empty - a Kaggle dataset that was never attached, or the wrong checkout.
if not os.path.isdir(os.path.join(SRC, "video_memory")):
    raise RuntimeError(
        f"""no video_memory package found under {SRC!r}
  REPO_URL   = {REPO_URL!r}
  SOURCE_DIR = {SOURCE_DIR!r}
  cwd        = {os.getcwd()!r}
Set REPO_URL to the GitHub repo, or SOURCE_DIR to a checkout / Kaggle dataset."""
    )

sys.path.insert(0, SRC)
import video_memory
print("imported video_memory from:", os.path.dirname(video_memory.__file__))

# Record which commit is running. The resolver's behaviour differs between
# branches, so any number you report is only meaningful next to a commit.
REPO_ROOT = os.path.dirname(SRC)
if os.path.isdir(os.path.join(REPO_ROOT, ".git")):
    for label, spec in (("branch", "--abbrev-ref"), ("commit", "--short")):
        r = subprocess.run(["git", "-C", REPO_ROOT, "rev-parse", spec, "HEAD"],
                           capture_output=True, text=True)
        print(f"git {label}: {r.stdout.strip() or '?'}")

## 4. Sanity check — run the CPU test batteries first

If these fail, stop: the bug is in logic, not in the model wiring, and it is far
cheaper to fix it locally than on a GPU box. This cell raises rather than warns,
because a red suite means every number produced below is suspect.

In [ ]:
TESTS = os.path.join(REPO_ROOT, "tests")
if os.path.isdir(TESTS):
    failed = []
    for t in ("test_resolve.py", "test_cascade.py",
              "test_query.py", "test_carry_forward.py"):
        p = os.path.join(TESTS, t)
        if not os.path.exists(p):
            continue
        print("==", t)
        if subprocess.run([sys.executable, p], check=False).returncode != 0:
            failed.append(t)
    print("\nfailing suites:", failed or "none")
    if failed:
        raise RuntimeError(
            f"{failed} failed - stop here. The bug is in CPU logic, not model "
            "wiring, and it is far cheaper to fix locally than on a GPU box."
        )
else:
    print("tests/ not found next to src/ - skipping (fine if you uploaded src only)")

## 5. Get a video

Any `.mp4` works mechanically, but **the footage has to match the prompt bank or
the run tells you nothing.** `perception/prompt_bank.py` is a closed,
first-person indoor vocabulary:

| relation | objects |
|---|---|
| `HOLDS` | cup, phone, book, bottle, knife, laptop, bag, pen, nothing |
| `LOCATED_IN` | kitchen, bedroom, office, bathroom, living room, store, street, garden |
| `ACTIVITY` | cooking, walking, typing, reading, eating, cleaning, driving, talking |
| `NEAR` | table, chair, door, window, sink, refrigerator, television, bed |

Feed that a drone flyover or a sports clip and SigLIP returns near-zero on every
prompt — the same bimodal behaviour the synthetic clip produces. You get an empty
memory and no way to tell "pipeline broken" from "footage out of vocabulary".
Section 7 has a guard cell that checks for exactly this.

**Where to get matching footage, on a headless box:**

```bash
# Best match: EPIC-KITCHENS-100. Egocentric, kitchen-centric, continuous.
# CC BY-NC 4.0, public download script, no signed agreement and no approval wait
# (unlike Ego4D). One participant is plenty - omitting --participants pulls TBs.
git clone https://github.com/epic-kitchens/epic-kitchens-download-scripts.git
python epic-kitchens-download-scripts/epic_downloader.py \
    --videos --participants P01 --output-path ~/data

# Zero-friction alternative: any CC0 "POV cooking" / "first person kitchen" clip
# from Pexels or Pixabay. Direct https, no auth. Not benchmark-grade, but real
# pixels containing real objects, which is the whole point of this run.
wget -O ~/data/clip.mp4 "<direct .mp4 url>"
```

Then set `VIDEO_PATH` in section 1 and rerun from there.

Licensing note: EPIC-KITCHENS is **non-commercial with attribution**, so commit
the *pack*, not the footage. `pack.py` stores embeddings and assertions — no
pixels — so the clone-and-run reviewer story survives with a credit line in the
README.

Leaving `VIDEO_PATH` empty falls back to the synthetic clip below. That exists to
prove the pipeline runs end to end, **not** to produce meaningful numbers: it is
coloured rectangles, it is out of vocabulary by construction, and its
`escalation_rate` of 1.0 is a property of the input, not a measurement of the
cascade.

In [ ]:
import numpy as np, cv2

SYNTHETIC_PATH = os.path.join(WORKDIR, "synthetic.mp4")


def make_synthetic_clip(path=None, seconds=60, fps=10):
    """A crude scripted clip: a coloured 'scene' that changes every 15s, with a
    moving square. Enough to exercise decode -> tag -> intervalize -> resolve."""
    path = path or SYNTHETIC_PATH
    h, w = 240, 320
    writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    scenes = [(30, 60, 90), (90, 60, 30), (40, 90, 60), (70, 70, 120)]
    for i in range(seconds * fps):
        t = i / fps
        bg = scenes[int(t // 15) % len(scenes)]
        frame = np.full((h, w, 3), bg, dtype=np.uint8)
        x = int((t * 25) % (w - 40))
        cv2.rectangle(frame, (x, 100), (x + 40, 140), (220, 220, 220), -1)
        writer.write(frame)
    writer.release()
    return path


SYNTHETIC = not VIDEO_PATH
if SYNTHETIC:
    VIDEO_PATH = make_synthetic_clip()
    print("using synthetic fallback clip - numbers from this run are NOT meaningful")
elif not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(
        f"VIDEO_PATH does not exist: {VIDEO_PATH!r}\n"
        "Set it to an absolute path, or leave it empty for the synthetic fallback."
    )

print("video:", VIDEO_PATH, f"| {os.path.getsize(VIDEO_PATH) / 1e6:.1f} MB")

## 6. Ingest

Stage 3 loads **lazily** — if no frame escalates, Qwen2-VL is never downloaded.

Watch the reported `escalation_rate`: that is the cascade's efficiency claim, and
section 10 is where it gets judged.

In [ ]:
from video_memory.ingest import ingest_video

meta = ingest_video(
    video_path=VIDEO_PATH,
    out_path=OUT_PACK,
    sample_fps=SAMPLE_FPS,
    tau=TAU,
    max_frames=MAX_FRAMES,
    stage3_enabled=STAGE3,
    min_duration=MIN_DURATION,
)

import json
print(json.dumps(meta, indent=2))

## 7. Inspect the resolved memory

In [ ]:
from video_memory.pack import read_pack

pack = read_pack(OUT_PACK)
assertions = pack["assertions"]
print(f"{len(assertions)} resolved assertions\n")

def fmt(t):
    if t == float("inf"):
        return "  open "
    return f"{int(t // 60):02d}:{t % 60:05.2f}"

for a in sorted(assertions, key=lambda x: (x.relation, x.interval.start))[:40]:
    print(f"{fmt(a.interval.start)} -> {fmt(a.interval.end)}  "
          f"{a.relation:<11} {a.object:<20} conf={a.confidence:.2f}  "
          f"{a.interval.closure.value:<13} n_frames={len(a.evidence.frame_ids)}")

In [ ]:
# Is the footage in vocabulary? SigLIP's sigmoid head is bimodal: on in-domain
# input the winning prompt scores high, on out-of-domain input EVERY prompt
# scores near zero. Both produce a small memory, so without this check an
# out-of-vocabulary clip is indistinguishable from a broken pipeline.
#
# This is a diagnostic, not a quality bar - a low number means "wrong footage for
# this prompt bank", which is the closed-vocabulary limitation working as
# documented, not a bug.
if assertions:
    confs = sorted((a.confidence for a in assertions), reverse=True)
    top, median = confs[0], confs[len(confs) // 2]
    print(f"confidence: max={top:.3f}  median={median:.3f}  n={len(confs)}")
    if top < 0.15:
        print(
            "\nWARNING: every assertion scored near zero.\n"
            "  This is the out-of-vocabulary signature, not a pipeline failure.\n"
            "  The prompt bank is first-person indoor domestic/office (section 5).\n"
            "  Change the footage, not the code - and do not report escalation\n"
            "  rate or x_realtime from this run, they measure nothing."
        )
    else:
        print("\nfootage is in vocabulary - the cascade numbers below are meaningful")
else:
    print("no assertions at all: check the WARNING above, or MIN_DURATION/MAX_FRAMES")

if SYNTHETIC:
    print("\nreminder: this is the synthetic clip. Out of vocabulary by "
          "construction. escalation_rate=1.0 here is a property of the input.")

In [ ]:
# Sanity property: no two single-valued assertions may overlap.
from collections import defaultdict
from video_memory.perception.prompt_bank import exclusive_relations, DEFAULT_BANK

ex = exclusive_relations(DEFAULT_BANK)
slots = defaultdict(list)
for a in assertions:
    if a.relation in ex:
        slots[(a.subject, a.relation)].append(a)

violations = 0
for key, items in slots.items():
    items.sort(key=lambda x: x.interval.start)
    for p, n in zip(items, items[1:]):
        if n.interval.start < p.interval.end - 1e-9:
            violations += 1
            print("OVERLAP:", key, p.object, p.interval, "vs", n.object, n.interval)

print("single-valued overlap violations:", violations, "(must be 0)")

## 8. Coverage of the timeline

How much of the video does the memory actually make a claim about? Low coverage
is not automatically bad — it means the system declined to assert things it was
unsure of — but it must be reported honestly alongside accuracy.

In [ ]:
duration = meta["video_duration_s"]
by_rel = defaultdict(float)
for a in assertions:
    end = min(a.interval.end, duration)
    by_rel[a.relation] += max(0.0, end - a.interval.start)

for rel, covered in sorted(by_rel.items()):
    print(f"{rel:<12} {covered:7.1f}s / {duration:.1f}s  ({100*covered/duration:5.1f}%)")

## 9. Take the pack with you

The pack is small (embeddings dominate, and those are ~0.6 KB/frame). Commit it
to the repo so the query side runs with no GPU and no dataset access — and note
that it contains **no pixels**, only embeddings and assertions, which is what
makes committing footage-derived data licence-safe.

In [ ]:
# On Kaggle the zip is what you download from the Output pane. On a server it is
# what you scp back:  scp you@box:~/vmem-out/pack.zip .
archive = shutil.make_archive(os.path.join(WORKDIR, "pack"), "zip", OUT_PACK)
print("wrote", archive, f"{os.path.getsize(archive) / 1e6:.2f} MB")
for f in sorted(os.listdir(OUT_PACK)):
    print(f"  {f:<20} {os.path.getsize(os.path.join(OUT_PACK, f)) / 1e3:8.1f} KB")

## 10. Ablation: cascade off (SigLIP only)

Reruns with `stage3_enabled=False` to get the A-noVLM arm. Comparing the two
packs is what turns "the cascade saves compute" into a measured number rather
than a design claim — and it is the arm that can falsify H3.

In [ ]:
meta_no_vlm = ingest_video(
    video_path=VIDEO_PATH,
    out_path=os.path.join(WORKDIR, "pack_no_vlm"),
    sample_fps=SAMPLE_FPS,
    tau=TAU,
    max_frames=MAX_FRAMES,
    stage3_enabled=False,
    min_duration=MIN_DURATION,
)

print("with VLM   :", meta["cascade"], f"x_realtime={meta['x_realtime']}")
print("without VLM:", meta_no_vlm["cascade"], f"x_realtime={meta_no_vlm['x_realtime']}")

# H3 lives or dies here. escalation_rate is the fraction of frames routed to the
# VLM; at 1.0 the cascade saved nothing and the compute claim is unsupported.
rate = meta["cascade"].get("escalation_rate")
if rate is not None:
    print(f"\nescalation_rate = {rate:.3f}")
    if rate > 0.9:
        print("  -> near 1.0: the cascade delivered no saving on this footage.\n"
              "     If this holds on in-vocabulary video, H3 is falsified and the\n"
              "     README should say so rather than defend the design.")
    else:
        print(f"  -> {100 * (1 - rate):.0f}% of frames avoided the VLM. Report this\n"
              "     next to the accuracy cost, never alone.")